## Objetivo

Esse trabalho busca entender o comportamento de vendas de uma operação de chocolates que atua em múltiplos países por meio de uma equipe de vendedores, identificando fatores que afetam a receita. Para isso deseja-se responder às seguintes perguntas:

1.	Quais produtos geram maior receita e quais geram maior volume de caixas? São os mesmos?
2.	Como a receita se distribui ao longo dos meses do período coberto?
3.	O preço médio por caixa varia entre países?
4.	Os vendedores de maior receita vendem mais volume ou vendem produtos mais caros?
5.	Existe concentração de receita em poucos produtos ou vendedores?


## Escolha da Base

Para responder aos questionamentos levantados, uma base que parece se adequar é a Chocolate Sales Data de Atharva Soundankar, encontrada no Kaggle. Com licença MIT, ou seja, permite uso, modificação e redistribuição, essa base contem registros de vendas de chocolate incluindo nome dos produtos, quantidade de caixas vendidas, valor, vendedor e data da venda, assim como o país em que ela ocorreu. 

Link da base: https://www.kaggle.com/datasets/atharvasoundankar/chocolate-sales?resource=download

## Coleta dos dados

A base no Kaggle encontra-se armazenada em um único arquivo CSV, com 14 kB, 1.094 linhas e 6 colunas, por isso a coleta foi feita através do upload do arquivo para o Databricks.

In [0]:
%sql

-- Criando os schemas bronze, silver e gold para armazenar as tabelas em cada etapa de tratamento
CREATE SCHEMA IF NOT EXISTS workspace.bronze;
CREATE SCHEMA IF NOT EXISTS workspace.silver;
CREATE SCHEMA IF NOT EXISTS workspace.gold;

-- Criando uma pasta para armazenar os arquivos brutos
CREATE VOLUME IF NOT EXISTS workspace.bronze.arquivos_brutos;

## Modelagem

Considerando o objetivo do trabalho e a base de dados escolhida, entende-se que existe um fato central (a venda) e diversas dimensões no entorno dele, de forma que o modelo a ser aplicado é o Star Schema.

## Pipeline ETL

#### Camada Bronze

Para salvar os dados brutos em um Dataframe, foi utilizado o argumento inferSchema como false, visando preservar o formato original dos dados, como esperado da camada Bronze.

In [0]:
## Salvando dados brutos da camada bronze em um dataframe
caminho = "/Volumes/workspace/bronze/arquivos_brutos/chocolate_sales.csv"

df = (spark.read
      .option("header", "true")
      .option("inferSchema", "false")
      .csv(caminho))

display(df)

Sales Person,Country,Product,Date,Amount,Boxes Shipped
Jehu Rudeforth,UK,Mint Chip Choco,04-Jan-22,"$5,320",180
Van Tuxwell,India,85% Dark Bars,01-Aug-22,"$7,896",94
Gigi Bohling,India,Peanut Butter Cubes,07-Jul-22,"$4,501",91
Jan Morforth,Australia,Peanut Butter Cubes,27-Apr-22,"$12,726",342
Jehu Rudeforth,UK,Peanut Butter Cubes,24-Feb-22,"$13,685",184
Van Tuxwell,India,Smooth Sliky Salty,06-Jun-22,"$5,376",38
Oby Sorrel,UK,99% Dark & Pure,25-Jan-22,"$13,685",176
Gunar Cockshoot,Australia,After Nines,24-Mar-22,"$3,080",73
Jehu Rudeforth,New Zealand,50% Dark Bites,20-Apr-22,"$3,990",59
Brien Boise,Australia,99% Dark & Pure,04-Jul-22,"$2,835",102


Foi necessário alterar os nomes das colunas 'Sales Person' e 'Boxes Shipped' para 'Sales_Person' e 'Boxes_Shipped' respectivamente, pois o Delta Lake não aceita nomes de coluna com espaços. Além disso, foram adicionadas as colunas de data da ingestão e origem para rastreabilidade.

Importante evidenciar que nenhum valor foi alterado nessa etapa, a renomeação das colunas citadas é uma exigência técnica do formato de armazenamento, não uma transformação de conteúdo de forma que o conteúdo bruto esperado na camada bronze foi mantido.

In [0]:
## Renomeando colunas e criando colunas de controle para gravar como tabela Delta
from pyspark.sql.functions import current_timestamp, lit

df_bronze = (df
    .withColumnRenamed("Sales Person", "Sales_Person")
    .withColumnRenamed("Boxes Shipped", "Boxes_Shipped")
    .withColumn("data_ingestao", current_timestamp())
    .withColumn("arquivo_origem", lit("chocolate_sales.csv"))
)

display(df_bronze)

Sales_Person,Country,Product,Date,Amount,Boxes_Shipped,data_ingestao,arquivo_origem
Jehu Rudeforth,UK,Mint Chip Choco,04-Jan-22,"$5,320",180,2026-09-27T14:25:32.505Z,chocolate_sales.csv
Van Tuxwell,India,85% Dark Bars,01-Aug-22,"$7,896",94,2026-09-27T14:25:32.505Z,chocolate_sales.csv
Gigi Bohling,India,Peanut Butter Cubes,07-Jul-22,"$4,501",91,2026-09-27T14:25:32.505Z,chocolate_sales.csv
Jan Morforth,Australia,Peanut Butter Cubes,27-Apr-22,"$12,726",342,2026-09-27T14:25:32.505Z,chocolate_sales.csv
Jehu Rudeforth,UK,Peanut Butter Cubes,24-Feb-22,"$13,685",184,2026-09-27T14:25:32.505Z,chocolate_sales.csv
Van Tuxwell,India,Smooth Sliky Salty,06-Jun-22,"$5,376",38,2026-09-27T14:25:32.505Z,chocolate_sales.csv
Oby Sorrel,UK,99% Dark & Pure,25-Jan-22,"$13,685",176,2026-09-27T14:25:32.505Z,chocolate_sales.csv
Gunar Cockshoot,Australia,After Nines,24-Mar-22,"$3,080",73,2026-09-27T14:25:32.505Z,chocolate_sales.csv
Jehu Rudeforth,New Zealand,50% Dark Bites,20-Apr-22,"$3,990",59,2026-09-27T14:25:32.505Z,chocolate_sales.csv
Brien Boise,Australia,99% Dark & Pure,04-Jul-22,"$2,835",102,2026-09-27T14:25:32.505Z,chocolate_sales.csv


In [0]:
## Gravando a bronze
(df_bronze.write
    .mode("overwrite")
    .saveAsTable("workspace.bronze.vendas_raw"))

Aqui é importante perceber que o pipeline opera em modo de carga completa (full load): cada execução relê integralmente o arquivo de origem e sobrescreve as tabelas de todas as camadas. 

Essa estratégia foi escolhida pois estamos trabalhando com um arquivo único que não recebe atualizações incrementais e, por conta dessa escolha, o campo data_ingestao registra o momento da carga mais recente, e não o histórico de ingestões. Em um cenário com a possibilidade de chegada de novos dados, a estratégia adequada seria a carga incremental, onde apenas os registros novos são adicionados na base e a data_ingestao pode ser usada para distinguir as levas.

##### Verificação da Tabela Gravada

Nessa etapa, deseja-se verificar se todas as linhas esperadas foram gravadas na nova tabela e ver os primeiros dados registrados. 

In [0]:
%sql
SELECT COUNT(*) AS total_linhas FROM workspace.bronze.vendas_raw;

total_linhas
1094


In [0]:
%sql
SELECT * FROM workspace.bronze.vendas_raw LIMIT 10;

Sales_Person,Country,Product,Date,Amount,Boxes_Shipped,data_ingestao,arquivo_origem
Jehu Rudeforth,UK,Mint Chip Choco,04-Jan-22,"$5,320",180,2026-09-27T14:25:38.771Z,chocolate_sales.csv
Van Tuxwell,India,85% Dark Bars,01-Aug-22,"$7,896",94,2026-09-27T14:25:38.771Z,chocolate_sales.csv
Gigi Bohling,India,Peanut Butter Cubes,07-Jul-22,"$4,501",91,2026-09-27T14:25:38.771Z,chocolate_sales.csv
Jan Morforth,Australia,Peanut Butter Cubes,27-Apr-22,"$12,726",342,2026-09-27T14:25:38.771Z,chocolate_sales.csv
Jehu Rudeforth,UK,Peanut Butter Cubes,24-Feb-22,"$13,685",184,2026-09-27T14:25:38.771Z,chocolate_sales.csv
Van Tuxwell,India,Smooth Sliky Salty,06-Jun-22,"$5,376",38,2026-09-27T14:25:38.771Z,chocolate_sales.csv
Oby Sorrel,UK,99% Dark & Pure,25-Jan-22,"$13,685",176,2026-09-27T14:25:38.771Z,chocolate_sales.csv
Gunar Cockshoot,Australia,After Nines,24-Mar-22,"$3,080",73,2026-09-27T14:25:38.771Z,chocolate_sales.csv
Jehu Rudeforth,New Zealand,50% Dark Bites,20-Apr-22,"$3,990",59,2026-09-27T14:25:38.771Z,chocolate_sales.csv
Brien Boise,Australia,99% Dark & Pure,04-Jul-22,"$2,835",102,2026-09-27T14:25:38.771Z,chocolate_sales.csv



A partir da visualização dos 10 primeiros registros já é possível identificar que todas as colunas originais tem tipo de dado String, inclusive a coluna de data da venda (Date) e de quantidade de caixas vendidas (Boxes_Shipped). Além disso, podemos notar que a coluna de valor (Amount) é preenchida com um cifrão inicial e também está com tipo string. Todos esses pontos serão tratados na camada Silver.

In [0]:
from pyspark.sql.functions import collect_list, array_join, sort_array, count, countDistinct

df_raw = spark.table("workspace.bronze.vendas_raw")

def dominio_bronze(df, coluna):
    """Retorna os valores distintos de uma coluna, ordenados e concatenados."""
    return (df.select(coluna).distinct()
              .agg(array_join(sort_array(collect_list(coluna)), ", "))
              .first()[0])

for coluna in ["Country", "Product", "Sales_Person"]:
    qtd = df_raw.select(coluna).distinct().count()
    print(f"--- {coluna} | {qtd} valores distintos ---")
    print(dominio_bronze(df_raw, coluna))
    print()

--- Country | 6 valores distintos ---
Australia, Canada, India, New Zealand, UK, USA

--- Product | 22 valores distintos ---
50% Dark Bites, 70% Dark Bites, 85% Dark Bars, 99% Dark & Pure, After Nines, Almond Choco, Baker's Choco Chips, Caramel Stuffed Bars, Choco Coated Almonds, Drinking Coco, Eclairs, Fruit & Nut Bars, Manuka Honey Choco, Milk Bars, Mint Chip Choco, Orange Choco, Organic Choco Syrup, Peanut Butter Cubes, Raspberry Choco, Smooth Sliky Salty, Spicy Special Slims, White Choc

--- Sales_Person | 25 valores distintos ---
Andria Kimpton, Barr Faughny, Beverie Moffet, Brien Boise, Camilla Castle, Ches Bonnell, Curtice Advani, Dennison Crosswaite, Dotty Strutley, Gigi Bohling, Gunar Cockshoot, Husein Augar, Jan Morforth, Jehu Rudeforth, Kaine Padly, Karlen McCaffrey, Kelci Walkden, Madelene Upcott, Mallorie Waber, Marney O'Breen, Oby Sorrel, Rafaelita Blaksland, Roddy Speechley, Van Tuxwell, Wilone O'Kielt



Optei por não listar os possíveis valores das colunas categóricas Sales_Person e Product dentro da descrição do catálogo por conta da grande extensão de valores distintos em cada uma. Como estamos tratando de uma base estática, os valores observados podem ser considerados domínios de colunas, mas é importante diferenciar a lista dos valores observados do conjunto de valores admissíveis da coluna (domínio). Em caso de atualizações na base, seria muito provável que eventualmente houvesse a adição de novos vendedores ou produtos, por exemplo. Ou seja, a lista de valores observados nessas duas colunas não representa um conjunto fechado de valores que elas podem assumir na prática.

##### Catálogo do dados da tabela bronze

In [0]:
%sql
COMMENT ON TABLE workspace.bronze.vendas_raw IS
'Tabela com registros de vendas de chocolate ingeridos do arquivo chocolate_sales.csv (Kaggle, Chocolate Sales Data, licença MIT) sem qualquer transformação de conteúdo. Campos de origem mantidos como texto, preservando formatos originais e colunas de controle de ingestão acrescentadas. Grão: cada linha contem o registro de uma venda contida no arquivo de origem. Carga do tipo full load: cada execução relê o arquivo integralmente e sobrescreve a tabela; Total: 1.094 registros.';

ALTER TABLE workspace.bronze.vendas_raw ALTER COLUMN Sales_Person COMMENT
'Nome do vendedor responsável pela venda, como consta na origem. Formato: Texto. Renomeado de "Sales Person" por restrição do Delta Lake; 25 valores distintos observados.';

ALTER TABLE workspace.bronze.vendas_raw ALTER COLUMN Country COMMENT
'País onde a venda foi realizada, como consta na origem. Formato: Texto. Domínio observado: Australia, Canada, India, New Zealand, UK, USA.';

ALTER TABLE workspace.bronze.vendas_raw ALTER COLUMN Product COMMENT
'Nome do produto de chocolate vendido, como consta na origem. Formato: Texto. 22 valores distintos observados.';

ALTER TABLE workspace.bronze.vendas_raw ALTER COLUMN Date COMMENT
'Data da venda em formato textual de origem (padrão dd-MMM-yy, ex.: 04-Jan-22).';

ALTER TABLE workspace.bronze.vendas_raw ALTER COLUMN Amount COMMENT
'Valor total da venda em formato textual de origem, incluindo cifrão e separador de milhar.';

ALTER TABLE workspace.bronze.vendas_raw ALTER COLUMN Boxes_Shipped COMMENT
'Quantidade de caixas expedidas na venda, em formato textual de origem. Renomeado de "Boxes Shipped" por restrição do Delta Lake; Domínio: Inteiros positivos.';

ALTER TABLE workspace.bronze.vendas_raw ALTER COLUMN data_ingestao COMMENT
'Metadado de controle: Timestamp da execução que gravou o registro. Registra o momento da ingestão mais recente e não o histórico de cargas, devido à carga full load. Gerado pelo pipeline.';

ALTER TABLE workspace.bronze.vendas_raw ALTER COLUMN arquivo_origem COMMENT
'Metadado de controle: Nome do arquivo de origem do registro. Valor fixo: chocolate_sales.csv. Gerado pelo pipeline para rastreabilidade de procedência.';

#### Camada Silver

Nessa etapa vamos inspecionar melhor o dado bruto e faremos as transformações necessárias para limpar a base.

In [0]:
df_raw = spark.table("workspace.bronze.vendas_raw")

# Como o Spark está enxergando cada coluna
df_raw.printSchema()

# Amostra dos valores problemáticos
display(df_raw.select("Date", "Amount", "Boxes_Shipped").limit(20))

root
 |-- Sales_Person: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Product: string (nullable = true)
 |-- Date: string (nullable = true)
 |-- Amount: string (nullable = true)
 |-- Boxes_Shipped: string (nullable = true)
 |-- data_ingestao: timestamp (nullable = true)
 |-- arquivo_origem: string (nullable = true)



Date,Amount,Boxes_Shipped
04-Jan-22,"$5,320",180
01-Aug-22,"$7,896",94
07-Jul-22,"$4,501",91
27-Apr-22,"$12,726",342
24-Feb-22,"$13,685",184
06-Jun-22,"$5,376",38
25-Jan-22,"$13,685",176
24-Mar-22,"$3,080",73
20-Apr-22,"$3,990",59
04-Jul-22,"$2,835",102


Confirmando o que já havia sido identificado, será necessário alterar os tipos das colunas Date, Amount e Boxes_Shipped para datetime, decimal e int respectivamente e remover o cifrão da coluna Amount. Além disso, optamos por traduzir para o português os nomes de cada coluna para facilitar a leitura.

In [0]:
## Verificando valores nulos e vazios
from pyspark.sql.functions import col, sum as _sum, when, count

colunas = ["Sales_Person", "Country", "Product", "Date", "Amount", "Boxes_Shipped"]

df_raw.select([
    _sum(when(col(c).isNull() | (col(c) == ""), 1).otherwise(0)).alias(c)
    for c in colunas
]).show()

+------------+-------+-------+----+------+-------------+
|Sales_Person|Country|Product|Date|Amount|Boxes_Shipped|
+------------+-------+-------+----+------+-------------+
|           0|      0|      0|   0|     0|            0|
+------------+-------+-------+----+------+-------------+



É possível ver que não há dados nulos ou vazios em nenhuma coluna do dataset.

In [0]:
## Aplicando transformações

from pyspark.sql.functions import regexp_replace, to_date, trim, initcap

df_silver = (df_raw
    # Renomeações finais para português sem espaços
    .withColumnRenamed("Sales_Person", "nome_vendedor")
    .withColumnRenamed("Country", "pais")
    .withColumnRenamed("Product", "produto")

    # Amount: removendo cifrão, vírgulas de milhar e espaços e convertendo para decimal
    .withColumn("valor_venda",
        regexp_replace(col("Amount"), r"[\$,\s]", "").cast("decimal(12,2)"))

    # Date: convertendo texto em data
    .withColumn("data_venda", to_date(col("Date"), "dd-MMM-yy"))

    # Boxes: convertendo texto para inteiro
    .withColumn("caixas_enviadas", col("Boxes_Shipped").cast("int"))

    # Padronização de texto: removendo espaços nas pontas e normalizando capitalização
    .withColumn("nome_vendedor", initcap(trim(col("nome_vendedor"))))
    .withColumn("pais", initcap(trim(col("pais"))))
    .withColumn("produto", trim(col("produto")))

    # Descartando colunas originais já convertidas
    .drop("Date", "Amount", "Boxes_Shipped")
)

display(df_silver)

nome_vendedor,pais,produto,data_ingestao,arquivo_origem,valor_venda,data_venda,caixas_enviadas
Jehu Rudeforth,Uk,Mint Chip Choco,2026-09-27T14:25:38.771Z,chocolate_sales.csv,5320.00,2022-01-04,180
Van Tuxwell,India,85% Dark Bars,2026-09-27T14:25:38.771Z,chocolate_sales.csv,7896.00,2022-08-01,94
Gigi Bohling,India,Peanut Butter Cubes,2026-09-27T14:25:38.771Z,chocolate_sales.csv,4501.00,2022-07-07,91
Jan Morforth,Australia,Peanut Butter Cubes,2026-09-27T14:25:38.771Z,chocolate_sales.csv,12726.00,2022-04-27,342
Jehu Rudeforth,Uk,Peanut Butter Cubes,2026-09-27T14:25:38.771Z,chocolate_sales.csv,13685.00,2022-02-24,184
Van Tuxwell,India,Smooth Sliky Salty,2026-09-27T14:25:38.771Z,chocolate_sales.csv,5376.00,2022-06-06,38
Oby Sorrel,Uk,99% Dark & Pure,2026-09-27T14:25:38.771Z,chocolate_sales.csv,13685.00,2022-01-25,176
Gunar Cockshoot,Australia,After Nines,2026-09-27T14:25:38.771Z,chocolate_sales.csv,3080.00,2022-03-24,73
Jehu Rudeforth,New Zealand,50% Dark Bites,2026-09-27T14:25:38.771Z,chocolate_sales.csv,3990.00,2022-04-20,59
Brien Boise,Australia,99% Dark & Pure,2026-09-27T14:25:38.771Z,chocolate_sales.csv,2835.00,2022-07-04,102


Na coluna produto optei por remover apenas espaços no início e final de cada nome, sem alterar a grafia do nome em si, uma vez que nomes de marcas tem grafia própria.

In [0]:
## Verificando quantidades pós transformações
total = df_silver.count()

df_silver.select(
    _sum(when(col("valor_venda").isNull(), 1).otherwise(0)).alias("nulos_valor"),
    _sum(when(col("data_venda").isNull(), 1).otherwise(0)).alias("nulos_data"),
    _sum(when(col("caixas_enviadas").isNull(), 1).otherwise(0)).alias("nulos_caixas")
).show()

print(f"Total de linhas: {total}")
print(f"Linhas distintas: {df_silver.distinct().count()}")

+-----------+----------+------------+
|nulos_valor|nulos_data|nulos_caixas|
+-----------+----------+------------+
|          0|         0|           0|
+-----------+----------+------------+

Total de linhas: 1094
Linhas distintas: 1094


Com isso é possível ter certeza que as conversões foram bem feitas em todas as células das colunas modificadas, pois nenhum nulo foi gerado e a quantidade de linhas se manteve. 

In [0]:
## Gravando a Silver
(df_silver.write
    .mode("overwrite")
    .saveAsTable("workspace.silver.vendas_limpas"))

##### Catálogo de dados da tabela Silver

In [0]:
%sql
COMMENT ON TABLE workspace.silver.vendas_limpas IS
'Camada Silver. Registros de vendas limpos, tipados e padronizados a partir de workspace.bronze.vendas_raw. Transformações aplicadas: remoção de símbolo monetário e separador de milhar, conversão de tipos, normalização de capitalização e remoção de espaços nas extremidades. Validação de conversão confirmou zero valores nulos nos três campos convertidos. Grão: cada linha contem o registro de uma venda contida no arquivo de origem. Total: 1.094 registros.';

ALTER TABLE workspace.silver.vendas_limpas ALTER COLUMN nome_vendedor COMMENT
'Nome do vendedor responsável pela venda. Formato: Texto. Padronizado com remoção de espaços nas extremidades e capitalização inicial. Linhagem: Bronze.Sales_Person. 25 valores distintos.';

ALTER TABLE workspace.silver.vendas_limpas ALTER COLUMN pais COMMENT
'País onde a venda foi realizada. Formato: Texto. Padronizado com remoção de espaços e capitalização inicial. Linhagem: Bronze.Country. Domínio: Australia, Canada, India, New Zealand, Uk, Usa.';

ALTER TABLE workspace.silver.vendas_limpas ALTER COLUMN produto COMMENT
'Nome do produto de chocolate vendido. Formato: Texto. Aplicada apenas remoção de espaços nas extremidades, a capitalização original foi preservada. Linhagem: Bronze.Product. 22 valores distintos.';

ALTER TABLE workspace.silver.vendas_limpas ALTER COLUMN data_venda COMMENT
'Data em que a venda foi realizada. Formato: data. Linhagem: convertido de Bronze.Date (texto, padrão dd-MMM-yy). Domínio: Datas válidas. Conversão sem geração de nulos.';

ALTER TABLE workspace.silver.vendas_limpas ALTER COLUMN valor_venda COMMENT
'Valor total da venda. Formato: Decimal com duas casas. Linhagem: convertido de Bronze.Amount mediante remoção de cifrão, separador de milhar e espaços. Domínio: Valores positivos.';

ALTER TABLE workspace.silver.vendas_limpas ALTER COLUMN caixas_enviadas COMMENT
'Quantidade de caixas expedidas na venda. Formato: Inteiro. Linhagem: convertido de Bronze.Boxes_Shipped. Domínio: Inteiros positivos.';

ALTER TABLE workspace.silver.vendas_limpas ALTER COLUMN data_ingestao COMMENT
'Metadado de controle propagado da camada Bronze. Timestamp da carga mais recente.';

ALTER TABLE workspace.silver.vendas_limpas ALTER COLUMN arquivo_origem COMMENT
'Metadado de controle propagado da camada Bronze. Nome do arquivo de origem.';

In [0]:
%sql

-- Verificando colunas da Silver
SELECT * FROM workspace.silver.vendas_limpas LIMIT 10;
DESCRIBE workspace.silver.vendas_limpas;

col_name,data_type,comment
nome_vendedor,string,Nome do vendedor responsável pela venda. Formato: Texto. Padronizado com remoção de espaços nas extremidades e capitalização inicial. Linhagem: Bronze.Sales_Person. 25 valores distintos.
pais,string,"País onde a venda foi realizada. Formato: Texto. Padronizado com remoção de espaços e capitalização inicial. Linhagem: Bronze.Country. Domínio: Australia, Canada, India, New Zealand, Uk, Usa."
produto,string,"Nome do produto de chocolate vendido. Formato: Texto. Aplicada apenas remoção de espaços nas extremidades, a capitalização original foi preservada. Linhagem: Bronze.Product. 22 valores distintos."
data_ingestao,timestamp,Metadado de controle propagado da camada Bronze. Timestamp da carga mais recente.
arquivo_origem,string,Metadado de controle propagado da camada Bronze. Nome do arquivo de origem.
valor_venda,"decimal(12,2)","Valor total da venda. Formato: Decimal com duas casas. Linhagem: convertido de Bronze.Amount mediante remoção de cifrão, separador de milhar e espaços. Domínio: Valores positivos."
data_venda,date,"Data em que a venda foi realizada. Formato: data. Linhagem: convertido de Bronze.Date (texto, padrão dd-MMM-yy). Domínio: Datas válidas. Conversão sem geração de nulos."
caixas_enviadas,int,Quantidade de caixas expedidas na venda. Formato: Inteiro. Linhagem: convertido de Bronze.Boxes_Shipped. Domínio: Inteiros positivos.


#### Camada Gold

Nessa etapa, já com os dados limpos, criamos as dimensões do modelo escolhido (Star Schema), com objetivo de viabilizar a análise de negócio dos dados visando responder às perguntas levantadas inicialmente.

In [0]:
## Criando as dimensões

from pyspark.sql.functions import monotonically_increasing_id, col

dim_produto = (df_silver
    .select("produto").distinct()
    .withColumn("sk_produto", monotonically_increasing_id())
    .select("sk_produto", col("produto").alias("nome_produto"))
)

dim_vendedor = (df_silver
    .select("nome_vendedor").distinct()
    .withColumn("sk_vendedor", monotonically_increasing_id())
    .select("sk_vendedor", "nome_vendedor")
)

dim_pais = (df_silver
    .select("pais").distinct()
    .withColumn("sk_pais", monotonically_increasing_id())
    .select("sk_pais", col("pais").alias("nome_pais"))
)

Cada dimensão foi construida a partir da lista de valores únicos de cada atributo e de uma chave substituta (surrogate key) que consiste em um número utilizado como identificador interno do modelo. Optei pela função monotonically_increasing_id(), que gera valores crescentes com saltos, visando evitar warning de performance por falta de partition (No Partition Defined for Window operation), observado ao tentar utilizar row_number(). No caso de uma base com 1094 registros, isso não seria um problema, mas para bases muito maiores, poderia representar um gargalo de performance, uma vez que uma janela sem particionamento geraria concentração de dados em uma única partição.

Importante evidenciar que essa escolha não prejudica a criação das chaves substitutas, umas vez que elas não precisam necessariamente ser sequenciais, precisam apenas de estabilidade e unicidade.

In [0]:
## Criando dimensão tempo a partir das datas informadas

from pyspark.sql.functions import (
    year, month, quarter, date_format, dayofweek, when, min as _min, max as _max
)

limites = df_silver.select(_min("data_venda"), _max("data_venda")).first()
print("Período coberto:", limites[0], "até", limites[1])

dim_tempo = (df_silver
    .select("data_venda").distinct()
    .withColumnRenamed("data_venda", "data")
    .withColumn("ano", year("data"))
    .withColumn("mes", month("data"))
    .withColumn("nome_mes", date_format("data", "MMMM"))
    .withColumn("trimestre", quarter("data"))
    .withColumn("dia_semana", date_format("data", "EEEE"))
    .withColumn("flag_fim_semana",
        when(dayofweek("data").isin(1, 7), True).otherwise(False))
)

display(dim_tempo.orderBy("data"))

Período coberto: 2022-01-03 até 2022-08-31


data,ano,mes,nome_mes,trimestre,dia_semana,flag_fim_semana
2022-01-03,2022,1,January,1,Monday,false
2022-01-04,2022,1,January,1,Tuesday,false
2022-01-05,2022,1,January,1,Wednesday,false
2022-01-07,2022,1,January,1,Friday,false
2022-01-10,2022,1,January,1,Monday,false
2022-01-11,2022,1,January,1,Tuesday,false
2022-01-12,2022,1,January,1,Wednesday,false
2022-01-13,2022,1,January,1,Thursday,false
2022-01-14,2022,1,January,1,Friday,false
2022-01-17,2022,1,January,1,Monday,false


A base cobre um período de 8 meses de janeiro a agosto do ano de 2022, o que é uma limitação importante a considerar no momento da análise, pois não consegimos observar um ano completo e os meses faltantes (setembro a dezembro) são bastante relevantes no contexto do negócio, já que espera-se um aumento na venda de chocolates em torno de diversas datas comemorativas como Haloween e Natal.

In [0]:
## Construindo tabela fato

fato_vendas = (df_silver
    .join(dim_produto, df_silver.produto == dim_produto.nome_produto)
    .join(dim_vendedor, "nome_vendedor")
    .join(dim_pais, df_silver.pais == dim_pais.nome_pais)
    .select(
        "sk_produto",
        "sk_vendedor",
        "sk_pais",
        col("data_venda"),
        col("valor_venda"),
        col("caixas_enviadas")
    )
)

display(fato_vendas)

sk_produto,sk_vendedor,sk_pais,data_venda,valor_venda,caixas_enviadas
5,6,2,2022-01-04,5320.00,180
3,20,5,2022-08-01,7896.00,94
1,12,5,2022-07-07,4501.00,91
1,8,0,2022-04-27,12726.00,342
1,6,2,2022-02-24,13685.00,184
6,20,5,2022-06-06,5376.00,38
7,1,2,2022-01-25,13685.00,176
13,2,0,2022-03-24,3080.00,73
20,6,4,2022-04-20,3990.00,59
7,23,0,2022-07-04,2835.00,102


In [0]:
## testando resultados das transformações pra garantir integridade

print("Linhas na Silver:", df_silver.count())
print("Linhas na fato:", fato_vendas.count())

soma_silver = df_silver.agg({"valor_venda": "sum"}).first()[0]
soma_fato = fato_vendas.agg({"valor_venda": "sum"}).first()[0]
print("Receita Silver:", soma_silver)
print("Receita Fato:  ", soma_fato)

Linhas na Silver: 1094
Linhas na fato: 1094
Receita Silver: 6183625.00
Receita Fato:   6183625.00


Assim verificamos que não houve duplicadas nas linhas, que nenhum valor se perdeu e que os valores das receitas não foram modificados nas etapas anteriores, de forma que podemos prosseguir com a gravação das tabelas da camada Gold.

In [0]:
## Gravando na Gold

dim_produto.write.mode("overwrite").saveAsTable("workspace.gold.dim_produto")
dim_vendedor.write.mode("overwrite").saveAsTable("workspace.gold.dim_vendedor")
dim_pais.write.mode("overwrite").saveAsTable("workspace.gold.dim_pais")
dim_tempo.write.mode("overwrite").saveAsTable("workspace.gold.dim_tempo")
fato_vendas.write.mode("overwrite").saveAsTable("workspace.gold.fato_vendas")

spark.sql("SHOW TABLES IN workspace.gold").show()


+--------+------------+-----------+
|database|   tableName|isTemporary|
+--------+------------+-----------+
|    gold|    dim_pais|      false|
|    gold| dim_produto|      false|
|    gold|   dim_tempo|      false|
|    gold|dim_vendedor|      false|
|    gold| fato_vendas|      false|
+--------+------------+-----------+



##### Catálogo de dados da tabela Gold

In [0]:
%sql
-- ---------- FATO ----------
COMMENT ON TABLE workspace.gold.fato_vendas IS
'Tabela fato do esquema estrela. Contém as métricas de negócio e as chaves substitutas para as dimensões produto, vendedor, país e tempo. Grão: uma linha por venda registrada. Linhagem: workspace.silver.vendas_limpas.';

ALTER TABLE workspace.gold.fato_vendas ALTER COLUMN sk_produto COMMENT
'Chave substituta do produto. Formato: Inteiro longo. Relaciona-se a dim_produto.sk_produto. Gerada pelo pipeline; valores únicos e não sequenciais.';

ALTER TABLE workspace.gold.fato_vendas ALTER COLUMN sk_vendedor COMMENT
'Chave substituta do vendedor. Formato: Inteiro longo. Relaciona-se a dim_vendedor.sk_vendedor. Gerada pelo pipeline; valores únicos e não sequenciais.';

ALTER TABLE workspace.gold.fato_vendas ALTER COLUMN sk_pais COMMENT
'Chave substituta do país. Formato: Inteiro longo. Relaciona-se a dim_pais.sk_pais. Gerada pelo pipeline; valores únicos e não sequenciais.';

ALTER TABLE workspace.gold.fato_vendas ALTER COLUMN data_venda COMMENT
'Data da venda. Formato: Data. Relaciona-se a dim_tempo.data. Domínio: 2022-01-03 a 2022-08-31. Linhagem: Silver.data_venda, sem transformação adicional.';

ALTER TABLE workspace.gold.fato_vendas ALTER COLUMN valor_venda COMMENT
'Valor total da venda. Formato: Decimal com duas casas. Domínio observado: valores positivos. Linhagem: Silver.valor_venda, sem transformação adicional.';

ALTER TABLE workspace.gold.fato_vendas ALTER COLUMN caixas_enviadas COMMENT
'Quantidade de caixas expedidas. Formato: Inteiro. Linhagem: Silver.caixas_enviadas, sem transformação adicional.';

-- ---------- DIMENSÕES ----------
COMMENT ON TABLE workspace.gold.dim_produto IS
'Dimensão produto. Grão: um registro por produto distinto do catálogo. 22 registros totais. Linhagem: valores distintos de workspace.silver.vendas_limpas.produto.';

ALTER TABLE workspace.gold.dim_produto ALTER COLUMN sk_produto COMMENT
'Chave substituta do produto. Formato: Inteiro longo, único, não sequencial. Gerada pelo pipeline.';

ALTER TABLE workspace.gold.dim_produto ALTER COLUMN nome_produto COMMENT
'Nome do produto de chocolate. Formato: Texto. Domínio: 22 valores distintos observados.';
--
COMMENT ON TABLE workspace.gold.dim_vendedor IS
'Dimensão vendedor. Grão: um registro por vendedor distinto. 25 registros totais. Linhagem: valores distintos de workspace.silver.vendas_limpas.nome_vendedor.';

ALTER TABLE workspace.gold.dim_vendedor ALTER COLUMN sk_vendedor COMMENT
'Chave substituta do vendedor. Formato: Inteiro longo, único, não sequencial. Gerada pelo pipeline.';

ALTER TABLE workspace.gold.dim_vendedor ALTER COLUMN nome_vendedor COMMENT
'Nome do vendedor. Formato: Texto. Domínio: 25 valores distintos observados.';
--
COMMENT ON TABLE workspace.gold.dim_pais IS
'Dimensão país. Grão: um registro por país distinto. 6 registros. Linhagem: valores distintos de workspace.silver.vendas_limpas.pais.';

ALTER TABLE workspace.gold.dim_pais ALTER COLUMN sk_pais COMMENT
'Chave substituta do país. Formato: Inteiro longo, único, não sequencial. Gerada pelo pipeline.';

ALTER TABLE workspace.gold.dim_pais ALTER COLUMN nome_pais COMMENT
'Nome do país onde a venda foi realizada. Formato: Texto. Domínio: Australia, Canada, India, New Zealand, Uk, Usa.';
--
COMMENT ON TABLE workspace.gold.dim_tempo IS
'Dimensão tempo. Grão: um registro por data distinta presente na base. Construída a partir do intervalo de datas de workspace.silver.vendas_limpas. Cobre o período de 2022-01-03 a 2022-08-31.';

ALTER TABLE workspace.gold.dim_tempo ALTER COLUMN data COMMENT
'Data de ocorrência da venda. Chave da dimensão. Formato: Data. Relaciona-se a fato_vendas.data_venda. Domínio: 2022-01-03 a 2022-08-31.';

ALTER TABLE workspace.gold.dim_tempo ALTER COLUMN ano COMMENT
'Ano da data. Formato: Inteiro. Domínio: 2022.';

ALTER TABLE workspace.gold.dim_tempo ALTER COLUMN mes COMMENT
'Número do mês. Formato: Inteiro. Domínio: 1 a 8.';

ALTER TABLE workspace.gold.dim_tempo ALTER COLUMN nome_mes COMMENT
'Nome do mês por extenso, em inglês. Formato: Texto. Domínio: January a August. Derivado de data.';

ALTER TABLE workspace.gold.dim_tempo ALTER COLUMN trimestre COMMENT
'Trimestre do ano. Formato: Inteiro. Domínio: 1 a 3.';

ALTER TABLE workspace.gold.dim_tempo ALTER COLUMN dia_semana COMMENT
'Dia da semana por extenso, em inglês. Formato: Texto. Domínio: Monday a Sunday. Derivado de data.';

ALTER TABLE workspace.gold.dim_tempo ALTER COLUMN flag_fim_semana COMMENT
'Indicador booleano de fim de semana. Verdadeiro para sábado e domingo, falso para os demais dias. Derivado de data.';

### Catálogo de dados completo

Para visualizar o catálogo de dados completo, podemos usar as consultas abaixo, sendo a primeira referente à descrição das tabelas e a segunda referente às colunas.

In [0]:
%sql
-- Catálogo de tabelas
SELECT table_schema, table_name, comment
FROM workspace.information_schema.tables
WHERE table_schema IN ('bronze','silver','gold')
ORDER BY table_schema, table_name;

table_schema,table_name,comment
bronze,vendas_raw,"Tabela com registros de vendas de chocolate ingeridos do arquivo chocolate_sales.csv (Kaggle, Chocolate Sales Data, licença MIT) sem qualquer transformação de conteúdo. Campos de origem mantidos como texto, preservando formatos originais e colunas de controle de ingestão acrescentadas. Grão: cada linha contem o registro de uma venda contida no arquivo de origem. Carga do tipo full load: cada execução relê o arquivo integralmente e sobrescreve a tabela; Total: 1.094 registros."
gold,dim_pais,Dimensão país. Grão: um registro por país distinto. 6 registros. Linhagem: valores distintos de workspace.silver.vendas_limpas.pais.
gold,dim_produto,Dimensão produto. Grão: um registro por produto distinto do catálogo. 22 registros totais. Linhagem: valores distintos de workspace.silver.vendas_limpas.produto.
gold,dim_tempo,Dimensão tempo. Grão: um registro por data distinta presente na base. Construída a partir do intervalo de datas de workspace.silver.vendas_limpas. Cobre o período de 2022-01-03 a 2022-08-31.
gold,dim_vendedor,Dimensão vendedor. Grão: um registro por vendedor distinto. 25 registros totais. Linhagem: valores distintos de workspace.silver.vendas_limpas.nome_vendedor.
gold,fato_vendas,"Tabela fato do esquema estrela. Contém as métricas de negócio e as chaves substitutas para as dimensões produto, vendedor, país e tempo. Grão: uma linha por venda registrada. Linhagem: workspace.silver.vendas_limpas."
silver,vendas_limpas,"Camada Silver. Registros de vendas limpos, tipados e padronizados a partir de workspace.bronze.vendas_raw. Transformações aplicadas: remoção de símbolo monetário e separador de milhar, conversão de tipos, normalização de capitalização e remoção de espaços nas extremidades. Validação de conversão confirmou zero valores nulos nos três campos convertidos. Grão: cada linha contem o registro de uma venda contida no arquivo de origem. Total: 1.094 registros."


In [0]:
%sql
-- Catálogo de colunas
SELECT table_schema, table_name, column_name, data_type, comment
FROM workspace.information_schema.columns
WHERE table_schema IN ('bronze','silver','gold')
ORDER BY table_schema, table_name, ordinal_position;

table_schema,table_name,column_name,data_type,comment
bronze,vendas_raw,Sales_Person,STRING,"Nome do vendedor responsável pela venda, como consta na origem. Formato: Texto. Renomeado de ""Sales Person"" por restrição do Delta Lake; 25 valores distintos observados."
bronze,vendas_raw,Country,STRING,"País onde a venda foi realizada, como consta na origem. Formato: Texto. Domínio observado: Australia, Canada, India, New Zealand, UK, USA."
bronze,vendas_raw,Product,STRING,"Nome do produto de chocolate vendido, como consta na origem. Formato: Texto. 22 valores distintos observados."
bronze,vendas_raw,Date,STRING,"Data da venda em formato textual de origem (padrão dd-MMM-yy, ex.: 04-Jan-22)."
bronze,vendas_raw,Amount,STRING,"Valor total da venda em formato textual de origem, incluindo cifrão e separador de milhar."
bronze,vendas_raw,Boxes_Shipped,STRING,"Quantidade de caixas expedidas na venda, em formato textual de origem. Renomeado de ""Boxes Shipped"" por restrição do Delta Lake; Domínio: Inteiros positivos."
bronze,vendas_raw,data_ingestao,TIMESTAMP,"Metadado de controle: Timestamp da execução que gravou o registro. Registra o momento da ingestão mais recente e não o histórico de cargas, devido à carga full load. Gerado pelo pipeline."
bronze,vendas_raw,arquivo_origem,STRING,Metadado de controle: Nome do arquivo de origem do registro. Valor fixo: chocolate_sales.csv. Gerado pelo pipeline para rastreabilidade de procedência.
gold,dim_pais,sk_pais,LONG,"Chave substituta do país. Formato: Inteiro longo, único, não sequencial. Gerada pelo pipeline."
gold,dim_pais,nome_pais,STRING,"Nome do país onde a venda foi realizada. Formato: Texto. Domínio: Australia, Canada, India, New Zealand, Uk, Usa."


## Qualidade de dados

Nessa etapa, as avaliações de completude, unicidade, consistência, acurácia e outliers foram feitas sobre a tabela silver, pois nela já temos a tipagem correta, a limpeza da base já foi feita e os dados já estão consultáveis. Além disso, no caso trabalhado, nenhum registro precisou ser descartado durante o tratamento, de forma que a tabela silver representa todo o conteúdo inserido originalmente.

#### Completude

In [0]:
## Gerando a tabela para verificar a completude dos dados

from pyspark.sql.functions import col, when, sum as _sum

df_raw = spark.table("workspace.bronze.vendas_raw")
df_silver = spark.table("workspace.silver.vendas_limpas")

pares = [
    ("Sales_Person",  "nome_vendedor"),
    ("Country",       "pais"),
    ("Product",       "produto"),
    ("Date",          "data_venda"),
    ("Amount",        "valor_venda"),
    ("Boxes_Shipped", "caixas_enviadas"),
]

total_b = df_raw.count()
total_s = df_silver.count()

def nulos(df, coluna):
    return df.select(
        _sum(when(col(coluna).isNull() | (col(coluna).cast("string") == ""), 1)
             .otherwise(0))
    ).first()[0]

linhas = ["| Atributo (Bronze) | Atributo (Silver) | Nulos Bronze | Nulos Silver | Completude |",
          "|---|---|---|---|---|"]

for cb, cs in pares:
    nb, ns = nulos(df_raw, cb), nulos(df_silver, cs)
    pct = 100 * (total_s - ns) / total_s
    linhas.append(f"| {cb} | {cs} | {nb} | {ns} | {pct:.2f}% |")

tabela = "\n".join(linhas)
print(f"Registros — Bronze: {total_b} | Silver: {total_s}\n")
print(tabela)

Registros — Bronze: 1094 | Silver: 1094

| Atributo (Bronze) | Atributo (Silver) | Nulos Bronze | Nulos Silver | Completude |
|---|---|---|---|---|
| Sales_Person | nome_vendedor | 0 | 0 | 100.00% |
| Country | pais | 0 | 0 | 100.00% |
| Product | produto | 0 | 0 | 100.00% |
| Date | data_venda | 0 | 0 | 100.00% |
| Amount | valor_venda | 0 | 0 | 100.00% |
| Boxes_Shipped | caixas_enviadas | 0 | 0 | 100.00% |



Em markdown para facilitar a visualização, temos:

| Atributo (Bronze) | Atributo (Silver) | Nulos Bronze | Nulos Silver | Completude |
|---|---|---|---|---|
| Sales_Person | nome_vendedor | 0 | 0 | 100.00% |
| Country | pais | 0 | 0 | 100.00% |
| Product | produto | 0 | 0 | 100.00% |
| Date | data_venda | 0 | 0 | 100.00% |
| Amount | valor_venda | 0 | 0 | 100.00% |
| Boxes_Shipped | caixas_enviadas | 0 | 0 | 100.00% |


Dessa forma é possível ver na tabela gerada que não há ocorrência de nulos ou vazios em nenhuma das colunas da base.

#### Unicidade

In [0]:
%sql
SELECT COUNT(*) AS total,
       COUNT(DISTINCT CONCAT(nome_vendedor, produto, data_venda, valor_venda)) AS combinacoes_distintas
FROM workspace.silver.vendas_limpas;

total,combinacoes_distintas
1094,1094


A partir da consulta acima, podemos concluir que não há registros duplicados na base, ou seja, não temos casos de mesma data, produto, valor e vendedor, mas é importante notar que dois registros gêmeos nessa base ainda poderiam se tratar de vendas distintas com mesmas condições. A falta de um identificador de cada venda impossibilita que afirmemos se linhas repetidas são erros na base ou não, o que evidencia uma limitação real da base.

#### Consistência e acurácia

In [0]:
%sql

SELECT
  MIN(data_venda) AS data_min,
  MAX(data_venda) AS data_max,
  SUM(CASE WHEN valor_venda <= 0 THEN 1 ELSE 0 END) AS valores_nao_positivos,
  SUM(CASE WHEN caixas_enviadas <= 0 THEN 1 ELSE 0 END) AS caixas_nao_positivas
FROM workspace.silver.vendas_limpas;

data_min,data_max,valores_nao_positivos,caixas_nao_positivas
2022-01-03,2022-08-31,0,0


Pela consulta acima, as datas estão no formato correto e dentro de um intervalo razoável. Como vimos que não houveram registros anulados na conversão para tipo data, também temos que nenhuma data registrada era inválida (como 32/09). Além disso, todos os valores de vendas e quantidades de caixas vendidas respeitam o domínio, ou seja, não temos valores negativos ou zerados, que seriam considerados erros na base uma vez que não faria sentido uma venda com tais características.

#### Outliers

Aqui optei por criar a coluna preco_medio_caixa que permite comparar produtos caros de baixo volume com produtos baratos de alto volume a partir da razão entre valor de venda e a quantidade de caixas enviadas.

In [0]:
%sql
SELECT
  MIN(preco_medio_caixa) AS minimo,
  PERCENTILE(preco_medio_caixa, 0.25) AS q1,
  PERCENTILE(preco_medio_caixa, 0.50) AS mediana,
  PERCENTILE(preco_medio_caixa, 0.75) AS q3,
  MAX(preco_medio_caixa) AS maximo,
  AVG(preco_medio_caixa) AS media
FROM ( SELECT v.valor_venda / v.caixas_enviadas AS preco_medio_caixa
       FROM workspace.silver.vendas_limpas v)

minimo,q1,mediana,q3,maximo,media
0.0135135135135,14.592563291139225,36.141843971631204,79.16035661944312,4291.0000000000000,105.72020449539501024


Podemos ver um forte indício de assimetria à direita, pois temos uma média de 105,72 contra mediana de 36,14. Além disso, temos um valor máximo muito acima do limite superior calculado tradicionalmente a partir do intervalo interquartil, nesse caso o limite superior fica em torno de 176 e o máximo observado foi de 4.291. Esse comportamento assimétrico talvez seja explicado por vendas de produtos com alto valor e pouca quantidade de caixas, o que inflacionaria o preço médio da caixa, mas dado o contexto de negócio, o valor máximo encontrado está bastante acima do esperado para um preço unitário de caixa de chocolate, mesmo para um produto caro. Vale observar o top 10 das vendas com maiores valores.

Outro ponto destacável é o mínimo de 0,01. No contexto do problema, mesmo em cenários de grandes promoções, esse valor seria muito abaixo do esperado para uma caixa de chocolate independentemente do produto, de forma que também vale investigar o top 10 das vendas com menores valores.

In [0]:
%sql
SELECT v.caixas_enviadas, v.valor_venda, v.valor_venda / v.caixas_enviadas AS preco_medio_caixa ,v.produto
FROM workspace.silver.vendas_limpas v 
ORDER BY 3 DESC LIMIT 10;

caixas_enviadas,valor_venda,preco_medio_caixa,produto
1,4291.00,4291.0000000000000,White Choc
2,6384.00,3192.0000000000000,Mint Chip Choco
3,6979.00,2326.3333333333333,Eclairs
2,4592.00,2296.0000000000000,White Choc
4,8939.00,2234.7500000000000,Fruit & Nut Bars
6,12586.00,2097.6666666666667,99% Dark & Pure
4,7413.00,1853.2500000000000,Mint Chip Choco
7,12586.00,1798.0000000000000,99% Dark & Pure
4,6587.00,1646.7500000000000,White Choc
6,9457.00,1576.1666666666667,Choco Coated Almonds


In [0]:
%sql
SELECT v.caixas_enviadas, v.valor_venda, v.valor_venda / v.caixas_enviadas AS preco_medio_caixa ,v.produto
FROM workspace.silver.vendas_limpas v 
ORDER BY preco_medio_caixa ASC LIMIT 10;

caixas_enviadas,valor_venda,preco_medio_caixa,produto
518,7.00,0.0135135135135,Smooth Sliky Salty
446,28.00,0.0627802690583,Peanut Butter Cubes
84,7.00,0.0833333333333,99% Dark & Pure
363,49.00,0.1349862258953,Choco Coated Almonds
181,63.00,0.3480662983425,Mint Chip Choco
223,112.00,0.5022421524664,Drinking Coco
97,49.00,0.5051546391753,Orange Choco
321,168.00,0.5233644859813,Fruit & Nut Bars
153,84.00,0.5490196078431,Caramel Stuffed Bars
105,63.00,0.6000000000000,After Nines


Analisando os últimos dados fica evidente a grande discrepância entre os valores mais altos e os mais baixos, mas trazendo também os produtos vendidos foi possível notar que algumas marcas estão presentes em ambos os grupos, como 99% Dark & Pure, Choco Coated Almonds, Mint Chip Choco e Fruit & Nut Bars, ou seja, a variação dos valores unitários das caixas desses produtos é muito grande, de foma que a métrica do preço médio acaba perdendo o sentido. Vale investigar a amplitude de valores das demais marcas e a correlação entre valor_venda e caixas_enviadas.

In [0]:
%sql
SELECT v.produto,
       COUNT(*) AS vendas,
       MIN(v.valor_venda / v.caixas_enviadas) AS preco_min,
       ROUND(AVG(v.valor_venda / v.caixas_enviadas), 2) AS preco_medio,
       MAX(v.valor_venda / v.caixas_enviadas) AS preco_max
FROM workspace.silver.vendas_limpas v 
GROUP BY v.produto
ORDER BY preco_max DESC;

produto,vendas,preco_min,preco_medio,preco_max
White Choc,58,3.8450704225352,232.05,4291.0000000000000
Mint Chip Choco,45,0.3480662983425,161.58,3192.0000000000000
Eclairs,60,2.4640000000000,117.65,2326.3333333333333
Fruit & Nut Bars,50,0.5233644859813,107.15,2234.7500000000000
99% Dark & Pure,49,0.0833333333333,146.80,2097.6666666666667
Choco Coated Almonds,39,0.1349862258953,120.40,1576.1666666666667
Spicy Special Slims,54,1.0500000000000,119.89,1541.7500000000000
Organic Choco Syrup,52,1.5689655172414,91.18,1389.5000000000000
85% Dark Bars,50,0.6791044776119,103.13,1243.6666666666667
50% Dark Bites,60,1.5317286652079,111.75,1225.0000000000000


In [0]:
%sql
SELECT ROUND(CORR(valor_venda, caixas_enviadas), 4) AS correlacao
FROM workspace.silver.vendas_limpas;

correlacao
-0.0188


De fato podemos ver que há uma amplitude de preços muito grande em todos os produtos e a correlção encontrada de -0,0188 está muito abaixo do esperado para a relação observada, uma vez que espera-se que o valor da venda seja dado pela multiplicação entre o valor unitário da caixa e a quantidade de caixas vendidas, ou seja, uma relação fortemente linear supondo uma variação menor no valor unitário das caixas. No gráfico abaixo, podemos visualizar claramente a situação.

In [0]:
%sql
SELECT caixas_enviadas, valor_venda, produto
FROM workspace.silver.vendas_limpas
;

caixas_enviadas,valor_venda,produto
180,5320.00,Mint Chip Choco
94,7896.00,85% Dark Bars
91,4501.00,Peanut Butter Cubes
342,12726.00,Peanut Butter Cubes
184,13685.00,Peanut Butter Cubes
38,5376.00,Smooth Sliky Salty
176,13685.00,99% Dark & Pure
73,3080.00,After Nines
59,3990.00,50% Dark Bites
102,2835.00,99% Dark & Pure


Databricks visualization. Run in Databricks to view.

O gráfico de dispersão entre caixas_enviadas e valor_venda, evidencia visualmente a ausência de relação mencionada, pois podemos observar que os pontos se distribuem de forma homogênea pelo plano, sem tendência diagonal, confirmando o coeficiente de correlação de Pearson encontrado.

A segmentação por produto elimina ainda a hipótese de que essa ausência de relação seja causada por variações entre os produtos, pois se fosse o caso, a nuvem observada resultaria da sobreposição de 22 relações lineares distintas, que ficariam evidentes com a coloração dos pontos. Na prática, o gráfico mostra cores uniformemente dispersas, sem que qualquer produto ocupe faixa ou região preferencial do plano. 

Também é interessante notar que a amplitude de receita permanece constante ao longo de todo o eixo horizontal, de forma que vendas de grandes quantidades de caixas alcançam os mesmos valores que vendas de poucas caixas.

A partir disso, é possível concluir que os campos de receita e volume foram gerados de forma independente na fonte, ou seja, apesar da descrição do Kaggle afirmar agregação de varejistas reais, o comportamento dos dados indica geração sintética sem modelagem de preço. Dessa forma, a confiabilidade da base para análise de negócio está comprometida e torna-se interessante uma seção de análise das relações entre as demais colunas. 

#### Avaliação de independência entre colunas

###### Vendas vs. Vendedores

Visando identificar se as vendas também independem da coluna de vendedores, vamos verificar se a quantidade de vendas tem grande amplitude dentro das categorias, o que corresponderia ao comportamento esperado de que certos vendedores vendem bem mais que outros. Além disso, vamos observar se a quantidade de produtos distintos por vendedor corresponde ao comportamento esperado de que alguns vendedores teriam uma pequena variedade de produtos no seu portifólio por conta fatores como especialização profissional ou carteira específica.

In [0]:
%sql
SELECT v.nome_vendedor, COUNT(DISTINCT f.sk_produto) AS produtos_distintos, COUNT(*) AS vendas
FROM workspace.gold.fato_vendas f
JOIN workspace.gold.dim_vendedor v ON f.sk_vendedor = v.sk_vendedor
GROUP BY v.nome_vendedor ORDER BY vendas DESC;

nome_vendedor,produtos_distintos,vendas
Kelci Walkden,22,54
Brien Boise,21,53
Van Tuxwell,21,51
Beverie Moffet,20,50
Oby Sorrel,20,49
Dennison Crosswaite,21,49
Ches Bonnell,20,48
Gigi Bohling,20,47
Karlen Mccaffrey,21,47
Curtice Advani,21,46


Podemos ver que a coluna de vendedores também parece ser gerada sinteticamente, pois quase todos os vendedores vendem em torno de 20 dos 22 produtos observados e a amplitude de vendas varia de 32 a 54, ou seja, não há especialidade ou concentração, parece se tratar de um sorteio independente a cada linha.


##### Vendas vs. Países

Para verificar a relação entre as vendas e os países observados, vamos avaliar a quantidade de vendas e a receita média, observando se os dados traduzem o comportamento esperado de que certos países vendem e arrecadam bem mais que outros devido a fatores como poder aquisitivo populacional ou canal de distribuição.

In [0]:
%sql
-- A distribuição de vendas por país é uniforme demais?
SELECT p.nome_pais, COUNT(*) AS vendas, ROUND(AVG(f.valor_venda), 2) AS receita_media
FROM workspace.gold.fato_vendas f
JOIN workspace.gold.dim_pais p ON f.sk_pais = p.sk_pais
GROUP BY p.nome_pais ORDER BY vendas DESC;

nome_pais,vendas,receita_media
Australia,205,5548.13
India,184,5683.70
Usa,179,5784.07
Uk,178,5908.94
Canada,175,5502.28
New Zealand,173,5493.75


Aqui vemos que os 6 países tem pouca variação da quantidade de vendas e de receita média, ou seja, são muito parecidos entre si, o que é mais um indício de base artificial. Dessa forma, confirmamos que a base não tem estrutura conjunta entre colunas e parece não ser adequada para a análise de negócio desejada. 

#### Síntese da avaliação de qualidade

| Critério | Resultado |
|---|---|
| Completude | Sem problemas — 100% em todos os atributos |
| Unicidade | Sem duplicatas; limitação por ausência de identificador de transação |
| Consistência | Sem problemas — formatos e intervalos válidos |
| Acurácia | Sem valores impossíveis nos campos individuais |
| Outliers | Assimetria extrema na razão entre receita e volume |
| **Relação entre campos** | **Comprometida — campos gerados de forma independente** |


## Análise Final 


Esta seção apresenta as consultas e discussões referentes às perguntas formuladas na etapa de objetivo. Contudo, as perguntas 3 e 4 dependem da métrica de preço unitário, invalidada na etapa de verificação de qualidade, na qual vimos que valor_venda e caixas_enviadas tem baixíssima correlação na base observada. Dessa forma, não será possível respondê-las usando os dados analizados. 

Para as demais perguntas, ainda que as conclusões de negócio estejam comprometidas pela natureza da fonte, vale executar as consultas necessárias para respondê-las para validar o modelo dimensional construído na camada Gold, evidenciando as junções entre fato e dimensões do esquema estrela. Além disso, ainda podemos analisar os resultados sob a hipótese de que a base é sintética e não corresponde ao comportamento real do mercado, avaliando mais uma vez se isso se confirma nas observações.


#####Pergunta 1 

Quais produtos geram maior receita e quais geram maior volume de caixas? São os mesmos?

In [0]:
%sql
SELECT
  p.nome_produto,
  COUNT(*)                      AS qtd_vendas,
  SUM(f.valor_venda)            AS receita_total,
  SUM(f.caixas_enviadas)        AS caixas_total,
  RANK() OVER (ORDER BY SUM(f.valor_venda) DESC)     AS rank_receita,
  RANK() OVER (ORDER BY SUM(f.caixas_enviadas) DESC) AS rank_volume
FROM workspace.gold.fato_vendas f
JOIN workspace.gold.dim_produto p ON f.sk_produto = p.sk_produto
GROUP BY p.nome_produto
ORDER BY receita_total DESC;

nome_produto,qtd_vendas,receita_total,caixas_total,rank_receita,rank_volume
Smooth Sliky Salty,59,349692.00,8810,1,2
50% Dark Bites,60,341712.00,9792,2,1
White Choc,58,329147.00,8240,3,10
Peanut Butter Cubes,49,324842.00,8304,4,8
Eclairs,60,312445.00,8757,5,3
99% Dark & Pure,49,299796.00,8127,6,12
85% Dark Bars,50,299229.00,7793,7,14
Organic Choco Syrup,52,294700.00,7749,8,16
Spicy Special Slims,54,293454.00,8685,9,5
Mint Chip Choco,45,283969.00,8207,10,11


A consulta ordena os 22 produtos por receita acumulada e por volume de caixas, apresentando as duas posições lado a lado. O produto de maior receita foi Smooth Sliky Salty (349.692,00), enquanto o de maior volume foi o 50% Dark Bites (9.792 caixas). Mas é evidente que há grande divergência entre os ordenamentos, pois temos casos de produtos com muitas posições de diferença nos rankins, por exemplo, o White Choc ocupa a 3ª posição em receita e apenas a 10ª em volume, enquanto Caramel Stuffed Bars é 21º em receita e 4º em volume. 

Esse comportamento é consistente com o achado da etapa de qualidade, pois como valor_venda e caixas_enviadas não guardam relação funcional, os dois rankings derivam de sorteios independentes. Em uma base real, a divergência seria informativa, distinguindo produtos caros e pouco vendidos daqueles mais acessíveis com grande quantidade de vendas, mas com os daos observados, essa divergência não é interpretável como característica de portfólio.



##### Pergunta 2 

Como a receita se distribui ao longo dos meses do período coberto?

In [0]:
%sql
SELECT
  t.ano,
  t.mes,
  t.nome_mes,
  COUNT(*)               AS qtd_vendas,
  SUM(f.valor_venda)     AS receita_total,
  SUM(f.caixas_enviadas) AS caixas_total
FROM workspace.gold.fato_vendas f
JOIN workspace.gold.dim_tempo t ON f.data_venda = t.data
GROUP BY t.ano, t.mes, t.nome_mes
ORDER BY t.ano, t.mes;

ano,mes,nome_mes,qtd_vendas,receita_total,caixas_total
2022,1,January,154,896105.00,27535
2022,2,February,110,699377.00,18015
2022,3,March,131,749483.00,19561
2022,4,April,118,674051.00,21003
2022,5,May,135,752892.00,21856
2022,6,June,163,865144.00,26260
2022,7,July,149,803425.00,22876
2022,8,August,134,743148.00,19901


A consulta acima agrega receita, volume e quantidade de vendas por mês, mas se ve limitada pelo período curto coberto pela base e pelos meses faltantes relevantes no contexto. 

Analisando os meses que é possível observar, vemos que a receita mensal variou entre 674.051,00 (abril) e 896.105,00 (janeiro), apresentando uma amplitude de aproximadamente 33%, sem tendências de alta ou baixa ao longo da período. A ausência de relação entre a extensão do mês e o volume registrado reforça a interpretação anterior de que a variação observada é flutuação aleatória, e não efeito de calendário ou demanda.

##### Pergunta 5 

Existe concentração de receita em poucos produtos ou vendedores?

In [0]:
%sql
WITH receita_produto AS (
  SELECT p.nome_produto, SUM(f.valor_venda) AS receita
  FROM workspace.gold.fato_vendas f
  JOIN workspace.gold.dim_produto p ON f.sk_produto = p.sk_produto
  GROUP BY p.nome_produto
)
SELECT
  nome_produto,
  receita,
  ROUND(100.0 * receita / SUM(receita) OVER (), 2) AS pct_receita,
  ROUND(100.0 * SUM(receita) OVER (ORDER BY receita DESC) / SUM(receita) OVER (), 2) AS pct_acumulado
FROM receita_produto
ORDER BY receita DESC;

nome_produto,receita,pct_receita,pct_acumulado
Smooth Sliky Salty,349692.00,5.66,5.66
50% Dark Bites,341712.00,5.53,11.18
White Choc,329147.00,5.32,16.50
Peanut Butter Cubes,324842.00,5.25,21.76
Eclairs,312445.00,5.05,26.81
99% Dark & Pure,299796.00,4.85,31.66
85% Dark Bars,299229.00,4.84,36.50
Organic Choco Syrup,294700.00,4.77,41.26
Spicy Special Slims,293454.00,4.75,46.01
Mint Chip Choco,283969.00,4.59,50.60


In [0]:
%sql
WITH receita_vendedores AS (
  SELECT v.nome_vendedor, SUM(f.valor_venda) AS receita
  FROM workspace.gold.fato_vendas f
  JOIN workspace.gold.dim_vendedor v ON f.sk_vendedor = v.sk_vendedor
  GROUP BY v.nome_vendedor
)
SELECT
  nome_vendedor,
  receita,
  ROUND(100.0 * receita / SUM(receita) OVER (), 2) AS pct_receita,
  ROUND(100.0 * SUM(receita) OVER (ORDER BY receita DESC) / SUM(receita) OVER (), 2) AS pct_acumulado
FROM receita_vendedores
ORDER BY receita DESC

nome_vendedor,receita,pct_receita,pct_acumulado
Ches Bonnell,320901.00,5.19,5.19
Oby Sorrel,316645.00,5.12,10.31
Madelene Upcott,316099.00,5.11,15.42
Brien Boise,312816.00,5.06,20.48
Kelci Walkden,311710.00,5.04,25.52
Van Tuxwell,303149.00,4.90,30.42
Dennison Crosswaite,291669.00,4.72,35.14
Beverie Moffet,278922.00,4.51,39.65
Kaine Padly,266490.00,4.31,43.96
Marney O'breen,259742.00,4.20,48.16


In [0]:
%sql
WITH receita_paises  AS (
  SELECT p.nome_pais, SUM(f.valor_venda) AS receita
  FROM workspace.gold.fato_vendas f
  JOIN workspace.gold.dim_pais p ON f.sk_pais = p.sk_pais
  GROUP BY p.nome_pais
)
SELECT
  nome_pais,
  receita,
  ROUND(100.0 * receita / SUM(receita) OVER (), 2) AS pct_receita,
  ROUND(100.0 * SUM(receita) OVER (ORDER BY receita DESC) / SUM(receita) OVER (), 2) AS pct_acumulado
FROM receita_paises
ORDER BY receita DESC

nome_pais,receita,pct_receita,pct_acumulado
Australia,1137367.00,18.39,18.39
Uk,1051792.00,17.01,35.40
India,1045800.00,16.91,52.31
Usa,1035349.00,16.74,69.06
Canada,962899.00,15.57,84.63
New Zealand,950418.00,15.37,100.00


Na primeira consulta temos a participação individual e acumulada de cada produto na receita total, permitindo verificar a existência de concentração no padrão de Pareto. Os seis produtos de maior receita observada concentram 31,66% do faturamento, enquanto o esperado em uma operação real seria uma concentração significativa, com essa fatia de quase 1/3 do portfólio respondendo por algo próximo a 70% ou 80% da receita. Além disso, o resultado observado aproxima-se de uma distribuição uniforme, onde a participação acumulada cresce de forma quase proporcional ao número de itens, e as participações individuais decrescem suavemente, sem qualquer degrau que separe produtos líderes dos demais. 

Aplicando a mesma análise a vendedores e países encontramos um padrão equivalente: Entre os seis países, a participação na receita variou entre 15,37% e 18,39%, e temos que em uma distribuição perfeitamente uniforme teriamos 16,67% de participação na receita para cada país, ou seja, a distribuição observada é quase uniforme. Entre os 25 vendedores, a participação do primeiro colocado foi de 5,19% contra 4,00% da uniformidade teórica. 

Assim, podemos concluir que em nenhuma das três dimensões analisadas observou-se a concentração que caracteriza operações comerciais reais.

## Conclusão

O objetivo inicial era entender o comportamento de vendas de uma operação de chocolates e para isso foi construido um pipeline funcional, no qual os dados percorrem as três camadas, a modelagem dimensional está consistente e o teste de integridade confirma preservação de registros e receita entre Silver e Gold.

O que a etapa de análise revelou, porém, é que a fonte escolhida não sustenta conclusões de negócio. Na verificação de qualidade foi possível identificar a ausência de relação funcional entre receita e quantidade, amplitude implausível de preço unitário por produto, distribuição praticamente uniforme entre países e ausência total de especialização entre vendedores. Dessa forma, o conjunto de evidências indica fortemente a geração sintética sem modelagem de estrutura conjunta entre as colunas, apesar de a documentação da fonte declarar agregação a partir de varejistas reais.

Finalmente, as perguntas formuladas não são respondíveis com esta fonte e seria necessário buscar outra base de dados antes investir em dashboards ou modelos sobre ela. Nesse caso, o valor do pipeline não está nas respostas que produziu, mas em ter permitido descobrir a inadequação da fonte de forma sistemática e documentada.